In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import os

from ipywidgets import VBox, HBox, Dropdown, FloatText, IntText, Button, Output, Layout, HTML
from IPython.display import display, clear_output
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [2]:

# ============================================================
# 1. Load train/test
# ============================================================
train_path = "./data/split/train/train.csv"
test_path  = "./data/split/test/test.csv"

df_train = pd.read_csv(train_path)
df_test  = pd.read_csv(test_path)



In [3]:

# ============================================================
# 2. Load address → cluster mapping (mode của train)
# ============================================================
map_path = "./model/model6/address_cluster_map.csv"
if not os.path.isfile(map_path):
    raise FileNotFoundError("Thiếu file address_cluster_map.csv")

address_cluster_map = pd.read_csv(map_path)
addr_to_cluster = dict(zip(address_cluster_map["address"], address_cluster_map["cluster_mode"]))

# thêm cluster vào train (để vẽ scatter)
df_train["cluster"] = df_train["address"].map(addr_to_cluster)


In [4]:

# ============================================================
# 3. Load pre-trained model cho từng cluster
# ============================================================
models = {}
for cid in [0, 1, 2]:
    pt_path = f"./model/model6/cluster_{cid}.pt"
    if not os.path.isfile(pt_path):
        raise FileNotFoundError(f"Thiếu file {pt_path}")

    m = nn.Linear(2, 1)
    m.load_state_dict(torch.load(pt_path, map_location="cpu"))
    m.eval()
    models[cid] = m

print("✔ Loaded 3 models & address mapping.")


✔ Loaded 3 models & address mapping.


In [ ]:
# ============================================================
# 4. UI Components
# ============================================================
address_list = sorted(address_cluster_map["address"].unique().tolist())

# Address display name mapping
address_display_names = {
    "bình chánh": "Bình Chánh",
    "bình tân": "Bình Tân",
    "bình thạnh": "Bình Thạnh",
    "cần giờ": "Cần Giờ",
    "củ chi": "Củ Chi",
    "gò vấp": "Gò Vấp",
    "hóc môn": "Hóc Môn",
    "nhà bè": "Nhà Bè",
    "phú nhuận": "Phú Nhuận",
    "quận 1": "Quận 1",
    "quận 10": "Quận 10",
    "quận 11": "Quận 11",
    "quận 12": "Quận 12",
    "quận 2": "Quận 2",
    "quận 3": "Quận 3",
    "quận 4": "Quận 4",
    "quận 5": "Quận 5",
    "quận 6": "Quận 6",
    "quận 7": "Quận 7",
    "quận 8": "Quận 8",
    "quận 9": "Quận 9",
    "tân bình": "Tân Bình",
    "tân phú": "Tân Phú",
    "thủ đức": "Thủ Đức"
}

# Cluster name mapping
cluster_names = {0: "Nhóm giá thấp", 1: "Nhóm giá trung bình", 2: "Nhóm giá cao"}

# Input fields for prediction
address_options = [(address_display_names.get(addr, addr.title()), addr) for addr in address_list]
address_w = Dropdown(options=address_options, description="📍 Địa chỉ cần tra:", style={'description_width': '150px'}, layout=Layout(width="500px"))
area_w    = FloatText(value=50.0, description="📐 Diện tích (m²):", style={'description_width': '150px'}, layout=Layout(width="500px"))
bed_w     = IntText(value=2, description="🛏️ Số phòng ngủ:", style={'description_width': '150px'}, layout=Layout(width="500px"))

# Dropdown for visualization mode
viz_options = ["Tất cả nhóm giá", "Theo nhóm giá", "Theo địa chỉ cụ thể"]
viz_w = Dropdown(
    options=viz_options, 
    value="Tất cả nhóm giá", 
    description="📊 So sánh với:", 
    style={'description_width': '150px'}, 
    layout=Layout(width="500px")
)

# Secondary dropdown for cluster/address selection (initially disabled)
cluster_options = [(cluster_names[i], i) for i in [0, 1, 2]]
cluster_w = Dropdown(
    options=cluster_options, 
    value=0, 
    description="   ↳ Chọn nhóm:", 
    style={'description_width': '150px'}, 
    layout=Layout(width="472px", margin="0 0 0 30px"),
    disabled=True
)
specific_addr_w = Dropdown(
    options=address_options, 
    description="   ↳ Chọn địa chỉ:", 
    style={'description_width': '150px'}, 
    layout=Layout(width="472px", margin="0 0 0 30px"),
    disabled=True
)

# Dropdown for chart type
chart_options = [
    "Giá nhà theo diện tích & số phòng ngủ",
    "Giá nhà theo diện tích", 
    "Giá nhà theo số phòng ngủ"
]
chart_w = Dropdown(
    options=chart_options, 
    value="Giá nhà theo diện tích & số phòng ngủ", 
    description="📈 Loại biểu đồ:", 
    style={'description_width': '150px'}, 
    layout=Layout(width="500px")
)

run_btn   = Button(
    description="DỰ ĐOÁN GIÁ", 
    button_style="success", 
    layout=Layout(width="250px", height="50px"),
    style={'font_weight': 'bold', 'font_size': '16px'}
)
out       = Output()

# Handler to enable/disable secondary dropdowns based on viz_w selection
def on_viz_change(change):
    if change['new'] == "Theo nhóm giá":
        cluster_w.disabled = False
        specific_addr_w.disabled = True
    elif change['new'] == "Theo địa chỉ cụ thể":
        cluster_w.disabled = True
        specific_addr_w.disabled = False
    else:  # "Tất cả nhóm giá"
        cluster_w.disabled = True
        specific_addr_w.disabled = True

viz_w.observe(on_viz_change, names='value')

In [6]:
# ============================================================
# 5. Prediction Handler
# ============================================================
@run_btn.on_click
def _run(_):
    with out:
        clear_output(wait=True)

        sel_addr = address_w.value
        cluster_id = addr_to_cluster.get(sel_addr, 1)

        x = np.array([[area_w.value, bed_w.value]], dtype=np.float32)
        x_t = torch.tensor(x)

        # predict
        with torch.no_grad():
            y_pred = models[cluster_id](x_t).item()

        print(f"Địa chỉ: {address_display_names.get(sel_addr, sel_addr.title())}")
        print(f"Nhóm giá: {cluster_names[cluster_id]}")
        print(f"Giá nhà dự đoán: {y_pred:,.2f} tỷ VNĐ")
        print()

        # =====================================================
        # Calculate metrics for the reference cluster/address
        # =====================================================
        viz_mode = viz_w.value
        
        if viz_mode == "Tất cả nhóm giá":
            plot_df = df_train.copy()
            title_suffix = "Tất cả nhóm"
            ref_cluster_id = cluster_id
        elif viz_mode == "Theo nhóm giá":
            ref_cluster_id = cluster_w.value
            plot_df = df_train[df_train["cluster"] == ref_cluster_id]
            title_suffix = cluster_names[ref_cluster_id]
        else:  # "Theo địa chỉ cụ thể"
            ref_addr = specific_addr_w.value
            ref_cluster_id = addr_to_cluster.get(ref_addr, cluster_id)
            plot_df = df_train[df_train["address"] == ref_addr]
            title_suffix = f"{address_display_names.get(ref_addr, ref_addr.title())}"

        if plot_df.empty:
            print("Không có dữ liệu train cho lựa chọn này.")
            return

        # =====================================================
        # Visualization based on chart type
        # =====================================================
        chart_type = chart_w.value
        
        if chart_type == "Giá nhà theo diện tích & số phòng ngủ":
            fig, axes = plt.subplots(1, 2, figsize=(14, 5))
            
            # Plot 1: Price vs Area
            ax = axes[0]
            if viz_mode == "Tất cả nhóm giá":
                for cid in [0, 1, 2]:
                    clust_data = plot_df[plot_df["cluster"] == cid]
                    if not clust_data.empty:
                        ax.scatter(clust_data["area"], clust_data["price"], 
                                 label=cluster_names[cid], alpha=0.5)
            else:
                ax.scatter(plot_df["area"], plot_df["price"], c="blue", alpha=0.5, label="Actual output")
            
            area_range = np.linspace(plot_df["area"].min(), plot_df["area"].max(), 100)
            X_line = torch.tensor(
                np.column_stack([area_range, np.full_like(area_range, bed_w.value)]),
                dtype=torch.float32
            )
            with torch.no_grad():
                y_line = models[ref_cluster_id](X_line).numpy().ravel()
            
            ax.plot(area_range, y_line, "r-", linewidth=2, label=f"Model")
            ax.scatter([area_w.value], [y_pred], c="yellow", s=150, edgecolors="black", 
                      label=f"Predicted output", zorder=5)
            ax.set_xlabel("Diện tích (m²)")
            ax.set_ylabel("Giá nhà (tỷ VNĐ)")
            ax.set_title("Giá nhà theo diện tích")
            ax.legend()
            ax.grid(True, alpha=0.3)
            
            # Plot 2: Price vs Bedrooms
            ax = axes[1]
            if viz_mode == "Tất cả nhóm giá":
                for cid in [0, 1, 2]:
                    clust_data = plot_df[plot_df["cluster"] == cid]
                    if not clust_data.empty:
                        ax.scatter(clust_data["bedrooms"], clust_data["price"], 
                                 label=cluster_names[cid], alpha=0.5)
            else:
                ax.scatter(plot_df["bedrooms"], plot_df["price"], c="blue", alpha=0.5, label="Actual output")
            
            bed_range = np.arange(plot_df["bedrooms"].min(), plot_df["bedrooms"].max() + 1)
            X_line = torch.tensor(
                np.column_stack([np.full_like(bed_range, area_w.value, dtype=float), bed_range]),
                dtype=torch.float32
            )
            with torch.no_grad():
                y_line = models[ref_cluster_id](X_line).numpy().ravel()
            
            ax.plot(bed_range, y_line, "r-", linewidth=2, label=f"Model")
            ax.scatter([bed_w.value], [y_pred], c="yellow", s=150, edgecolors="black", 
                      label=f"Predicted output", zorder=5)
            ax.set_xlabel("Số phòng ngủ")
            ax.set_ylabel("Giá nhà (tỷ VNĐ)")
            ax.set_title("Giá nhà theo số phòng ngủ")
            ax.legend()
            ax.grid(True, alpha=0.3)
            
            fig.suptitle(f"Giá nhà so với {title_suffix}", fontsize=14, y=1.02)
            plt.tight_layout()
            plt.show()
            
        elif chart_type == "Giá nhà theo số phòng ngủ":
            # Price vs Bedrooms
            fig, ax = plt.subplots(figsize=(10, 6))
            
            if viz_mode == "Tất cả nhóm giá":
                for cid in [0, 1, 2]:
                    clust_data = plot_df[plot_df["cluster"] == cid]
                    if not clust_data.empty:
                        ax.scatter(clust_data["bedrooms"], clust_data["price"], 
                                 label=cluster_names[cid], alpha=0.5)
            else:
                ax.scatter(plot_df["bedrooms"], plot_df["price"], c="blue", alpha=0.5, label="Actual output")
            
            bed_range = np.arange(plot_df["bedrooms"].min(), plot_df["bedrooms"].max() + 1)
            X_line = torch.tensor(
                np.column_stack([np.full_like(bed_range, area_w.value, dtype=float), bed_range]),
                dtype=torch.float32
            )
            with torch.no_grad():
                y_line = models[ref_cluster_id](X_line).numpy().ravel()
            
            ax.plot(bed_range, y_line, "r-", linewidth=2, label=f"Model (Area={area_w.value}m²)")
            ax.scatter([bed_w.value], [y_pred], c="yellow", s=150, edgecolors="black", 
                      label=f"Predicted output", zorder=5)
            
            ax.set_xlabel("Số phòng ngủ")
            ax.set_ylabel("Giá nhà (tỷ VNĐ)")
            ax.set_title(f"Giá nhà theo số phòng ngủ - {title_suffix}")
            ax.legend()
            ax.grid(True)
            plt.tight_layout()
            plt.show()
            
        else: 
            # Price vs Area
            fig, ax = plt.subplots(figsize=(10, 6))
            
            if viz_mode == "Tất cả nhóm giá":
                for cid in [0, 1, 2]:
                    clust_data = plot_df[plot_df["cluster"] == cid]
                    if not clust_data.empty:
                        ax.scatter(clust_data["area"], clust_data["price"], 
                                 label=cluster_names[cid], alpha=0.5)
            else:
                ax.scatter(plot_df["area"], plot_df["price"], c="blue", alpha=0.5, label="Actual output")

            area_range = np.linspace(plot_df["area"].min(), plot_df["area"].max(), 100)
            X_line = torch.tensor(
                np.column_stack([area_range, np.full_like(area_range, bed_w.value)]),
                dtype=torch.float32
            )
            with torch.no_grad():
                y_line = models[ref_cluster_id](X_line).numpy().ravel()

            ax.plot(area_range, y_line, "r-", linewidth=2, label=f"Model (Bedrooms={bed_w.value})")
            ax.scatter([area_w.value], [y_pred], c="yellow", s=150, edgecolors="black", 
                      label=f"Predicted output", zorder=5)

            ax.set_xlabel("Diện tích (m²)")
            ax.set_ylabel("Giá nhà (tỷ VNĐ)")
            ax.set_title(f"Giá nhà theo diện tích - {title_suffix}")
            ax.legend()
            ax.grid(True)
            plt.tight_layout()
            plt.show()

        X_plot = torch.tensor(plot_df[["area", "bedrooms"]].values, dtype=torch.float32)
        y_plot_true = plot_df["price"].values
        
        with torch.no_grad():
            y_plot_pred = models[ref_cluster_id](X_plot).numpy().flatten()
        
        r2 = r2_score(y_plot_true, y_plot_pred)
        mse = mean_squared_error(y_plot_true, y_plot_pred)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(y_plot_true, y_plot_pred)
        
        print(f"Model Performance ({title_suffix}):")
        print(f"  R² Score: {r2:.4f}")
        print(f"  MAE: {mae:.2f} tỷ VNĐ")
        print(f"  RMSE: {rmse:.2f} tỷ VNĐ")
        print(f"  MSE: {mse:.2f}")
        print()

In [7]:
# ============================================================
# 6. Display UI
# ============================================================
print("=" * 80)
print("                   🏠 HỆ THỐNG DỰ ĐOÁN GIÁ NHÀ TP.HCM 🏠")
print("=" * 80)
print()
print("📝 HƯỚNG DẪN SỬ DỤNG:")
print("   1. Chọn địa chỉ bạn muốn tra cứu giá nhà")
print("   2. Nhập diện tích và số phòng ngủ của căn nhà")
print("   3. Chọn cách hiển thị biểu đồ so sánh:")
print("      • 'Tất cả nhóm giá': Xem tất cả dữ liệu (mặc định)")
print("      • 'Theo nhóm giá': Chỉ xem nhóm giá cụ thể (thấp/trung bình/cao)")
print("      • 'Theo địa chỉ cụ thể': Chỉ xem dữ liệu của một địa chỉ")
print("   4. Chọn loại biểu đồ bạn muốn xem")
print("   5. Nhấn nút 'Dự đoán giá' để xem kết quả")
print()
print("=" * 80)
print()

display(VBox([
    address_w, 
    area_w, 
    bed_w,
    viz_w, 
    cluster_w, 
    specific_addr_w,
    chart_w,
    HBox([run_btn], layout=Layout(margin='20px 0 20px 150px')),
    out
], layout=Layout(padding="20px")))

                   🏠 HỆ THỐNG DỰ ĐOÁN GIÁ NHÀ TP.HCM 🏠

📝 HƯỚNG DẪN SỬ DỤNG:
   1. Chọn địa chỉ bạn muốn tra cứu giá nhà
   2. Nhập diện tích và số phòng ngủ của căn nhà
   3. Chọn cách hiển thị biểu đồ so sánh:
      • 'Tất cả nhóm giá': Xem tất cả dữ liệu (mặc định)
      • 'Theo nhóm giá': Chỉ xem nhóm giá cụ thể (thấp/trung bình/cao)
      • 'Theo địa chỉ cụ thể': Chỉ xem dữ liệu của một địa chỉ
   4. Chọn loại biểu đồ bạn muốn xem
   5. Nhấn nút 'Dự đoán giá' để xem kết quả


